# Biomarker Analysis Pipeline

Runs all cohort combinations:
- **Cohorts**: `all_ICI` (any-line ICI vs never-ICI), `first_line` (first-line ICI vs never-ICI)

Each run produces 3 sensitivity specs internally: stabilized ATE, stabilized ATT, and unweighted (noIPTW).

### Stages
1. **Propensity scores** — `ICI_LRs_all_ICI.py`, `ICI_LRs_first_line.py`
2. **IPTW datasets** — `generate_IPTW_df.py --cohort {cohort}`
3. **Cox models** — `run_IPTW_analysis.py --cohort {cohort}`

In [2]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
COHORTS = ['all_ICI', 'first_line']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            universal_newlines=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Propensity Score Generation

In [3]:
run_and_stream('all_ICI propensity', [sys.executable, 'ICI_LRs_all_ICI.py'])
run_and_stream('first_line propensity', [sys.executable, 'ICI_LRs_first_line.py']) 


--- all_ICI propensity ---
ICI (IO_START ∩ cohort):      6699
Never ICI (treated, no ICI):  8741
IO_START not in cohort:       2223
Figure(2000x500)

Done: all_ICI propensity

--- first_line propensity ---
First-line ICI (IO_START ∩ first-line ∩ cohort): 2801
ICI in IO_START but not first-line:              3898
Never ICI (treated, no ICI):                     8741
IO_START not in cohort:                          2223
Figure(2000x500)

Done: first_line propensity


## Stage 2: IPTW Dataset Generation

In [4]:
for cohort in COHORTS:
    run_and_stream(f'{cohort} IPTW dataset',
                   [sys.executable, 'generate_IPTW_df.py', '--cohort', cohort])


--- all_ICI IPTW dataset ---
[generate_IPTW_df] Cohort: all_ICI
[generate_IPTW_df] Propensity path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/all_ICI_propensity/w_30_day_buffer/
[generate_IPTW_df] Prediction data path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/all_ICI_prediction_data/
[generate_IPTW_df] Saved 7512 patients to /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/IPTW_ICI_interaction_runs_df_all_ICI.csv

Done: all_ICI IPTW dataset

--- first_line IPTW dataset ---
[generate_IPTW_df] Cohort: first_line
[generate_IPTW_df] Propensity path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/first_line_ICI_propensity/w_30_day_buffer/
[generate_IPTW_df] Prediction data path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/first_line_ICI_prediction_data/
[generate_IPTW_df] Saved 5829 patients t

## Stage 3: IPTW Cox Model Analysis

In [5]:
for cohort in COHORTS:
    run_and_stream(f'{cohort} Cox models',
                   [sys.executable, 'run_IPTW_analysis.py', '--cohort', cohort])


--- all_ICI Cox models ---
[run_IPTW_analysis] Cohort: all_ICI
[run_IPTW_analysis] Output: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/IPTW_runs_all_ICI/
[pan_cancer] Merged 2 rare cancer types into CANCER_TYPE_OTHER: CANCER_TYPE_LYMPHOMA, CANCER_TYPE_MYELOMA
[pan_cancer] ATE: N treated=2556, N control=4240 | ESS treated=1860, ESS control=3629
[pan_cancer] ATT: N treated=2556, N control=4240 | ESS treated=2556, ESS control=2060
[SKIN] Recalibrated propensity scores within subset (mean 0.867 -> 0.936)
[SKIN] ATE: N treated=414, N control=29 | ESS treated=414, ESS control=29
[SKIN] ATT: N treated=414, N control=29 | ESS treated=414, ESS control=29
[LUNG] Recalibrated propensity scores within subset (mean 0.576 -> 0.603)
[LUNG] ATE: N treated=755, N control=502 | ESS treated=754, ESS control=501
[LUNG] ATT: N treated=755, N control=502 | ESS treated=755, ESS control=499

Done: all_ICI Cox models

--- first_line Cox models ---
[run_IPTW_analysis] Coh

In [8]:
output_path = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/'
all_ICI_path = os.path.join(output_path, 'IPTW_runs_all_ICI/')
first_line_path = os.path.join(output_path, 'IPTW_runs_first_line/')

import pandas as pd
pc_df = pd.read_csv(os.path.join(first_line_path, 'pan_cancer_ATT_ICI_predictive_markers.csv'))
lung_df = pd.read_csv(os.path.join(first_line_path, 'LUNG_ATT_ICI_predictive_markers.csv'))
skin_df = pd.read_csv(os.path.join(first_line_path, 'SKIN_ATT_ICI_predictive_markers.csv'))

cols_to_select = ['marker', 'beta_markerxICI', 'p_markerxICI', 'FDR_markerxICI', 'classifier']

pc_hits = pc_df.loc[pc_df['significant_predictive'], cols_to_select]
pc_hits['cancer_specificity'] = 'pan_cancer'

lung_hits = lung_df.loc[lung_df['significant_predictive'], cols_to_select]
lung_hits['cancer_specificity'] = 'lung'

skin_hits = skin_df.loc[skin_df['significant_predictive'], cols_to_select]
skin_hits['cancer_specificity'] = 'skin'

complete_hits = pd.concat([pc_hits, lung_hits, skin_hits])

complete_hits.to_csv(os.path.join(first_line_path, 'compiled_hits.csv'),index=False)